<a href="https://colab.research.google.com/github/Rrricardi/P/blob/main/Create_data_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

# Asegurar que el directorio 'data/' exista
os.makedirs("data", exist_ok=True)

# Creamos una base de datos SQLite en data/
engine = create_engine("sqlite:///data/retail_store.db")


def build_database():
  print("1/4 Cargando CSVs de Kaggle...")
  df_train = pd.read_csv("/content/drive/MyDrive/Data/train.csv")
  df_stores = pd.read_csv("/content/drive/MyDrive/Data/stores.csv")
  df_oil = pd.read_csv("/content/drive/MyDrive/Data/oil.csv")

  # Formatear fechas
  df_train["date"] = pd.to_datetime(df_train["date"])
  df_oil["date"] = pd.to_datetime(df_oil["date"])

  print("2/4 Creando tabla de dimensiones de productos/familias...")
  # Kaggle agrupa por 'family' (33 familias de productos)
  families = df_train["family"].unique()
  np.random.seed(42)

  # Parametros simulados de inventario (costo, margen y lead time en dias)
  df_products = pd.DataFrame({
      "product_id": families,
      "product_name": families,
      "category": "Retail",
      "unit_cost": np.random.uniform(2.5, 30.0, size=len(families)).round(2),
      "lead_time_days": np.random.choice([3, 5, 7, 10], size=len(families)),
  })
  df_products["selling_price"] = (df_products["unit_cost"] * 1.30).round(2)

  print("3/4 Preparando tabla de ventas y petroleo...")
  # Rellenar valores nulos de petroleo mediante interpolacion lineal hacia adelante
  df_oil["dcoilwtico"] = df_oil["dcoilwtico"].ffill().bfill()

  print("4/4 Guardando tablas en SQLite...")
  df_products.to_sql(
      "dim_products", engine, if_exists="replace", index=False
  )
  df_stores.to_sql("dim_stores", engine, if_exists="replace", index=False)
  df_oil.to_sql("fact_oil_prices", engine, if_exists="replace", index=False)

  # Para que la carga sea rapida, guardamos las ventas
  df_train.to_sql(
      "fact_daily_sales",
      engine,
      if_exists="replace",
      index=False,
      chunksize=50000,
  )

  print("Base de datos lista en data/retail_store.db!")


if __name__ == "__main__":
  build_database()

1/4 Cargando CSVs de Kaggle...
2/4 Creando tabla de dimensiones de productos/familias...
3/4 Preparando tabla de ventas y petroleo...
4/4 Guardando tablas en SQLite...
Base de datos lista en data/retail_store.db!
